In [1]:
!pip install transformers accelerate bitsandbytes peft
!pip install huggingface_hub sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.7 MB/s eta 0:00:00


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    model,
    "/content"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.k_proj.l

In [4]:
merged_model = model.merge_and_unload()

In [5]:
merged_model.save_pretrained("/content/quantized/model-fp16")
tokenizer.save_pretrained("/content/quantized/model-fp16")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/quantized/model-fp16/tokenizer_config.json',
 '/content/quantized/model-fp16/chat_template.jinja',
 '/content/quantized/model-fp16/tokenizer.json')

In [6]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model_int8 = AutoModelForCausalLM.from_pretrained(
    "/content/quantized/model-fp16",
    quantization_config=bnb_config,
    device_map="auto"
)

model_int8.save_pretrained("/content/quantized/model-int8")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    "/content/quantized/model-fp16",
    quantization_config=bnb_config,
    device_map="auto"
)

model_int4.save_pretrained("/content/quantized/model-int4")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [18]:
!python convert_hf_to_gguf.py \
../quantized/model-fp16 \
--outfile ../quantized/model-fp16.gguf

INFO:hf-to-gguf:Loading model: model-fp16
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:hf-to-gguf:heuristics detected float16 tensor dtype, setting --outtype f16
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         t

In [8]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp


Cloning into 'llama.cpp'...
remote: Enumerating objects: 82637, done.
remote: Counting objects: 100% (205/205), done.
remote: Compressing objects: 100% (169/169), done.
remote: Total 82637 (delta 115), reused 36 (delta 36), pack-reused 82432 (from 3)
Receiving objects: 100% (82637/82637), 311.96 MiB | 14.79 MiB/s, done.
Resolving deltas: 100% (59396/59396), done.
Updating files: 100% (2436/2436), done.
/content/llama.cpp


In [9]:
!mkdir -p build && cd build && cmake .. && cmake --build . --config Release -j 2

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

In [12]:
!cp build/bin/llama-quantize .

In [20]:
!pwd

/content/llama.cpp


In [21]:
!./llama-quantize ../quantized/model-fp16.gguf ../quantized/model_q8_0.gguf q8_0

main: build = 8252 (b51819510)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '../quantized/model-fp16.gguf' to '../quantized/model_q8_0.gguf' as Q8_0
llama_model_loader: loaded meta data with 30 key-value pairs and 201 tensors from ../quantized/model-fp16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Model Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_mode

In [22]:
!du -sh /content/quantized/*

2.1G	/content/quantized/model-fp16
2.1G	/content/quantized/model-fp16.gguf
771M	/content/quantized/model-int4
1.2G	/content/quantized/model-int8
1.1G	/content/quantized/model_q8_0.gguf


In [23]:
!zip -r quantized.zip /content/quantized

  adding: content/quantized/ (stored 0%)
  adding: content/quantized/model_q8_0.gguf (deflated 4%)
  adding: content/quantized/model-int4/ (stored 0%)
  adding: content/quantized/model-int4/model.safetensors (deflated 14%)
  adding: content/quantized/model-int4/generation_config.json (deflated 29%)
  adding: content/quantized/model-int4/config.json (deflated 56%)
  adding: content/quantized/model-fp16/ (stored 0%)
  adding: content/quantized/model-fp16/model.safetensors (deflated 23%)
  adding: content/quantized/model-fp16/tokenizer_config.json (deflated 46%)
  adding: content/quantized/model-fp16/tokenizer.json (deflated 85%)
  adding: content/quantized/model-fp16/generation_config.json (deflated 29%)
  adding: content/quantized/model-fp16/config.json (deflated 49%)
  adding: content/quantized/model-fp16/chat_template.jinja (deflated 60%)
  adding: content/quantized/model-fp16.gguf (deflated 23%)
  adding: content/quantized/model-int8/ (stored 0%)
  adding: content/quantized/model-int